# Photo → Sketch (PACS) Domain Adaptation

7-class domain adaptation from Photo (source) to Sketch (target) using PACS dataset.

- **Task I**: CycleGAN (spatial) vs Spectral CycleGAN style transfer
- **Task II**: UDA benchmark — Source-only, CycleGAN, Spectral CycleGAN, CyCADA, FDA

In [ ]:
# ============================================================
# Setup — clone repo to local disk, mount Drive for saving results
# ============================================================
import os

# Clone project repo to local disk (data + utils all included)
if not os.path.exists('/content/Project2'):
    !git clone https://github.com/sophie99zyq/NTU-AdvancedCV-Project2.git /content/Project2

# CycleGAN repo
if not os.path.exists('pytorch-CycleGAN-and-pix2pix'):
    !git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix.git
    !pip install -q visdom dominate

# Mount Drive for saving results only
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/Project2')

import torch
import torch.utils.data as data_utils
import os
from torchvision import datasets, transforms
from torchvision.utils import save_image
from torch.utils.data import TensorDataset, DataLoader
from PIL import Image

from utils.data_utils import get_pacs, get_paired_loaders, get_unnormalized_transform
from utils.fft_utils import fda_transfer
from utils.classifiers import ResNet50Classifier, train_classifier, evaluate_classifier
from utils.viz_utils import show_image_grid, print_results_table
from utils.eval_utils import save_results
from utils.cyclegan_wrapper import get_cyclegan_train_cmd, get_cyclegan_test_cmd
from utils.spectral_cyclegan import save_low_freq_images, reconstruct_from_translated_low
from utils.cycada import train_cycada, translate_with_cycada

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

DATA_ROOT = '/content/Project2/data/pacs/pacs/images'
SAVE_DIR = '/content/drive/MyDrive/Project2/results/photo_sketch'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# === Quick Test Mode ===
# Set FAST_TEST = True to verify the notebook runs end-to-end (~5 min)
# Set FAST_TEST = False for full experiment
FAST_TEST = True

if FAST_TEST:
    CYCLEGAN_EPOCHS = 1
    CYCLEGAN_DECAY = 0
    CYCADA_EPOCHS = 1
    CLASSIFIER_EPOCHS = 2
    MAX_IMAGES = 100
    NUM_TEST = 50
else:
    CYCLEGAN_EPOCHS = 50
    CYCLEGAN_DECAY = 50
    CYCADA_EPOCHS = 50
    CLASSIFIER_EPOCHS = 20
    MAX_IMAGES = 5000
    NUM_TEST = 60000

print(f'FAST_TEST={FAST_TEST}: epochs={CYCLEGAN_EPOCHS}+{CYCLEGAN_DECAY}, images={MAX_IMAGES}')

## Load Data

In [ ]:
# Normalized datasets (for classifier training/evaluation)
photo_train = get_pacs(root=DATA_ROOT, domain='photo')
sketch_train = get_pacs(root=DATA_ROOT, domain='sketch')

source_loader, target_loader = get_paired_loaders(photo_train, sketch_train, batch_size=32)
source_test_loader = DataLoader(photo_train, batch_size=32, shuffle=False, num_workers=2)
target_test_loader = DataLoader(sketch_train, batch_size=32, shuffle=False, num_workers=2)

# Unnormalized datasets [0,1] (for CycleGAN / CyCADA / FDA)
unnorm_transform = get_unnormalized_transform(img_size=224)
photo_unnorm = datasets.ImageFolder(os.path.join(DATA_ROOT, 'photo'), transform=unnorm_transform)
sketch_unnorm = datasets.ImageFolder(os.path.join(DATA_ROOT, 'sketch'), transform=unnorm_transform)

source_loader_unnorm, target_loader_unnorm = get_paired_loaders(photo_unnorm, sketch_unnorm, batch_size=32)
source_loader_unnorm_seq = DataLoader(photo_unnorm, batch_size=32, shuffle=False, num_workers=2)

print(f"Photo train: {len(photo_train)}, Sketch train: {len(sketch_train)}")
print(f"Classes: {photo_train.classes}")
print(f"Num classes: {len(photo_train.classes)}")

## Task I: Style Transfer Comparison

### Save images for CycleGAN training

In [ ]:
def save_dataset_images(dataset, output_dir, max_images=None):
    """Save dataset images as individual files for CycleGAN."""
    os.makedirs(output_dir, exist_ok=True)
    n = len(dataset) if max_images is None else min(len(dataset), max_images)
    for i in range(n):
        img, label = dataset[i]
        save_image(img, os.path.join(output_dir, f'{i:05d}.jpg'))
    print(f"Saved {n} images to {output_dir}")

CYCLEGAN_DATA = '/content/cyclegan_data/photo2sketch'
save_dataset_images(photo_unnorm, os.path.join(CYCLEGAN_DATA, 'trainA'), max_images=MAX_IMAGES)
save_dataset_images(sketch_unnorm, os.path.join(CYCLEGAN_DATA, 'trainB'), max_images=MAX_IMAGES)


### Train Spatial CycleGAN

In [ ]:
cmd = get_cyclegan_train_cmd(
    dataroot=CYCLEGAN_DATA,
    name='photo2sketch_spatial',
    n_epochs=CYCLEGAN_EPOCHS, n_epochs_decay=CYCLEGAN_DECAY,
    load_size=256, crop_size=224,
    input_nc=3, output_nc=3,
    netG='resnet_9blocks'
)
print(f"Running: {cmd}")
!{cmd}

### Train Spectral CycleGAN

In [ ]:
# Decompose to low-freq / high-freq
save_low_freq_images(
    os.path.join(CYCLEGAN_DATA, 'trainA'),
    '/content/spectral_data/photo', beta=0.03
)
save_low_freq_images(
    os.path.join(CYCLEGAN_DATA, 'trainB'),
    '/content/spectral_data/sketch', beta=0.03
)

# Prepare low-freq data for spectral CycleGAN
SPECTRAL_CYCLEGAN_DATA = '/content/cyclegan_data/photo2sketch_spectral'
os.makedirs(os.path.join(SPECTRAL_CYCLEGAN_DATA, 'trainA'), exist_ok=True)
os.makedirs(os.path.join(SPECTRAL_CYCLEGAN_DATA, 'trainB'), exist_ok=True)
!cp /content/spectral_data/photo/low/* {SPECTRAL_CYCLEGAN_DATA}/trainA/
!cp /content/spectral_data/sketch/low/* {SPECTRAL_CYCLEGAN_DATA}/trainB/

cmd = get_cyclegan_train_cmd(
    dataroot=SPECTRAL_CYCLEGAN_DATA,
    name='photo2sketch_spectral',
    n_epochs=CYCLEGAN_EPOCHS, n_epochs_decay=CYCLEGAN_DECAY,
    load_size=256, crop_size=224,
    input_nc=3, output_nc=3,
    netG='resnet_9blocks'
)
print(f"Running: {cmd}")
!{cmd}

### Visualize CycleGAN Results

In [ ]:
import glob, shutil, os

# Prepare testA/testB for CycleGAN visualization test
for base_name in ['photo2sketch', 'photo2sketch_spectral']:
    base = f'/content/cyclegan_data/{base_name}'
    for split in ['trainA', 'trainB']:
        src_dir = os.path.join(base, split)
        dst_dir = os.path.join(base, split.replace('train', 'test'))
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.exists(src_dir):
            for f in sorted(glob.glob(f'{src_dir}/*.png') + glob.glob(f'{src_dir}/*.jpg'))[:50]:
                shutil.copy2(f, dst_dir)
            print(f"Copied {len(os.listdir(dst_dir))} images to {dst_dir}")

In [ ]:
# Generate translated images with spatial CycleGAN
test_cmd_spatial = get_cyclegan_test_cmd(
    dataroot=CYCLEGAN_DATA,
    name='photo2sketch_spatial',
    load_size=256, crop_size=224,
    input_nc=3, output_nc=3,
    netG='resnet_9blocks'
)
!{test_cmd_spatial}

# Generate translated images with spectral CycleGAN
test_cmd_spectral = get_cyclegan_test_cmd(
    dataroot=SPECTRAL_CYCLEGAN_DATA,
    name='photo2sketch_spectral',
    load_size=256, crop_size=224,
    input_nc=3, output_nc=3,
    netG='resnet_9blocks'
)
!{test_cmd_spectral}

# Reconstruct spectral results: translated low-freq + original high-freq
reconstruct_from_translated_low(
    './results/photo2sketch_spectral/test_latest/images',
    '/content/spectral_data/photo/high',
    '/content/spectral_reconstructed/photo2sketch'
)

# Load and visualize
to_tensor = transforms.ToTensor()

def load_images_from_dir(d, n=8, suffix='_fake_B.png'):
    imgs = []
    for f in sorted(os.listdir(d)):
        if f.endswith(suffix):
            imgs.append(to_tensor(Image.open(os.path.join(d, f)).convert('RGB')))
        if len(imgs) >= n:
            break
    return torch.stack(imgs) if imgs else None

original = torch.stack([photo_unnorm[i][0] for i in range(8)])
spatial = load_images_from_dir('./results/photo2sketch_spatial/test_latest/images', 8)
recon_dir = '/content/spectral_reconstructed/photo2sketch'
spectral_recon = torch.stack([
    to_tensor(Image.open(os.path.join(recon_dir, f)).convert('RGB'))
    for f in sorted(os.listdir(recon_dir))[:8]
]) if os.path.exists(recon_dir) else None

viz_dict = {'Original Photo': original}
if spatial is not None:
    viz_dict['Spatial CycleGAN -> Sketch'] = spatial
if spectral_recon is not None:
    viz_dict['Spectral CycleGAN -> Sketch'] = spectral_recon

show_image_grid(
    viz_dict,
    title='Task I: Photo -> Sketch Style Transfer',
    save_path=os.path.join(SAVE_DIR, 'task1_comparison.png')
)

## Task II: UDA Benchmark

### Source-only baseline

In [ ]:
results = {}

model_source = ResNet50Classifier(num_classes=7)
model_source = train_classifier(model_source, source_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc = evaluate_classifier(model_source, target_test_loader, device=device)
results['Source-only'] = acc

### CycleGAN UDA

In [ ]:
# Save all source images for CycleGAN translation
CYCLEGAN_TEST_DATA = '/content/cyclegan_data/photo2sketch_test'
save_dataset_images(photo_unnorm, os.path.join(CYCLEGAN_TEST_DATA, 'testA'), max_images=MAX_IMAGES)
save_dataset_images(sketch_unnorm, os.path.join(CYCLEGAN_TEST_DATA, 'testB'), max_images=MAX_IMAGES)

test_all_cmd = get_cyclegan_test_cmd(
    netG='resnet_9blocks',
    dataroot=CYCLEGAN_TEST_DATA,
    name='photo2sketch_spatial',
    load_size=256, crop_size=224,
    input_nc=3, output_nc=3
)
!{test_all_cmd} --num_test {min(NUM_TEST, len(photo_unnorm))}

# Load translated images and apply ImageNet normalization
imagenet_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
)
resize_and_norm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_normalize,
])

translated_imgs = []
result_dir = './results/photo2sketch_spatial/test_latest/images'
for f in sorted(os.listdir(result_dir)):
    if f.endswith('_fake_B.png'):
        img = resize_and_norm(Image.open(os.path.join(result_dir, f)).convert('RGB'))
        translated_imgs.append(img)

translated_imgs = torch.stack(translated_imgs[:len(photo_train)])
labels = torch.tensor([photo_train[i][1] for i in range(len(translated_imgs))])

cyclegan_dataset = TensorDataset(translated_imgs, labels)
cyclegan_loader = DataLoader(cyclegan_dataset, batch_size=32, shuffle=True, num_workers=2)

model_cyclegan = ResNet50Classifier(num_classes=7)
model_cyclegan = train_classifier(model_cyclegan, cyclegan_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc = evaluate_classifier(model_cyclegan, target_test_loader, device=device)
results['CycleGAN'] = acc

### Spectral CycleGAN UDA

In [ ]:
# Save test data for spectral CycleGAN
SPECTRAL_TEST_DATA = '/content/cyclegan_data/photo2sketch_spectral_test'

# Decompose test source images
save_low_freq_images(
    os.path.join(CYCLEGAN_TEST_DATA, 'testA'),
    '/content/spectral_test_data/photo', beta=0.03
)

# Prepare spectral test data
os.makedirs(os.path.join(SPECTRAL_TEST_DATA, 'testA'), exist_ok=True)
os.makedirs(os.path.join(SPECTRAL_TEST_DATA, 'testB'), exist_ok=True)
!cp /content/spectral_test_data/photo/low/* {SPECTRAL_TEST_DATA}/testA/
!cp /content/spectral_data/sketch/low/* {SPECTRAL_TEST_DATA}/testB/

test_spectral_cmd = get_cyclegan_test_cmd(
    netG='resnet_9blocks',
    dataroot=SPECTRAL_TEST_DATA,
    name='photo2sketch_spectral',
    load_size=256, crop_size=224,
    input_nc=3, output_nc=3
)
!{test_spectral_cmd} --num_test {min(NUM_TEST, len(photo_unnorm))}

# Reconstruct: translated low-freq + original high-freq
reconstruct_from_translated_low(
    './results/photo2sketch_spectral/test_latest/images',
    '/content/spectral_test_data/photo/high',
    '/content/spectral_reconstructed_all/photo2sketch'
)

# Load reconstructed images with ImageNet normalization
spectral_imgs = []
recon_dir = '/content/spectral_reconstructed_all/photo2sketch'
for f in sorted(os.listdir(recon_dir)):
    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
        img = resize_and_norm(Image.open(os.path.join(recon_dir, f)).convert('RGB'))
        spectral_imgs.append(img)

spectral_imgs = torch.stack(spectral_imgs[:len(photo_train)])
spectral_dataset = TensorDataset(spectral_imgs, labels[:len(spectral_imgs)])
spectral_loader = DataLoader(spectral_dataset, batch_size=32, shuffle=True, num_workers=2)

model_spectral = ResNet50Classifier(num_classes=7)
model_spectral = train_classifier(model_spectral, spectral_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc = evaluate_classifier(model_spectral, target_test_loader, device=device)
results['Spectral CycleGAN'] = acc

### CyCADA

In [ ]:
# Train CyCADA with unnormalized [0,1] images
# Need a source-only classifier trained on [0,1] images for task loss
source_loader_unnorm_paired, target_loader_unnorm_paired = get_paired_loaders(
    photo_unnorm, sketch_unnorm, batch_size=32
)

# Train source classifier on unnormalized images for CyCADA task loss
model_source_unnorm = ResNet50Classifier(num_classes=7)
source_unnorm_train_loader = DataLoader(photo_unnorm, batch_size=32, shuffle=True, num_workers=2)
model_source_unnorm = train_classifier(
    model_source_unnorm, source_unnorm_train_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device
)

G_s2t, G_t2s = train_cycada(
    source_loader_unnorm_paired, target_loader_unnorm_paired,
    classifier=model_source_unnorm,
    num_classes=7, in_channels=3,
    n_epochs=CYCADA_EPOCHS, device=device
)

# Translate source images
translated_data = translate_with_cycada(G_s2t, source_loader_unnorm_seq, device=device)
cycada_imgs = torch.cat([d[0] for d in translated_data])
cycada_labels = torch.cat([d[1] for d in translated_data])

# Apply ImageNet normalization for classifier training
cycada_imgs_norm = torch.stack([imagenet_normalize(img) for img in cycada_imgs])

cycada_dataset = TensorDataset(cycada_imgs_norm, cycada_labels)
cycada_loader = DataLoader(cycada_dataset, batch_size=32, shuffle=True, num_workers=2)

model_cycada = ResNet50Classifier(num_classes=7)
model_cycada = train_classifier(model_cycada, cycada_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc = evaluate_classifier(model_cycada, target_test_loader, device=device)
results['CyCADA'] = acc

### FDA (Fourier Domain Adaptation)

In [ ]:
# FDA works on [0,1] images, then we normalize for the classifier
fda_imgs = []
fda_labels = []

for (src_imgs, src_lbls), (tgt_imgs, _) in zip(source_loader_unnorm, target_loader_unnorm):
    bs = min(src_imgs.size(0), tgt_imgs.size(0))
    transferred = fda_transfer(src_imgs[:bs], tgt_imgs[:bs], beta=0.01)
    fda_imgs.append(transferred)
    fda_labels.append(src_lbls[:bs])

fda_imgs = torch.cat(fda_imgs)
fda_labels = torch.cat(fda_labels)

# Visualize FDA results
show_image_grid(
    {'Original Photo': torch.stack([photo_unnorm[i][0] for i in range(8)]),
     'FDA Transferred': fda_imgs[:8]},
    title='FDA: Photo -> Sketch',
    save_path=os.path.join(SAVE_DIR, 'fda_comparison.png')
)

# Apply ImageNet normalization for classifier training
fda_imgs_norm = torch.stack([imagenet_normalize(img) for img in fda_imgs])

fda_dataset = TensorDataset(fda_imgs_norm, fda_labels)
fda_loader = DataLoader(fda_dataset, batch_size=32, shuffle=True, num_workers=2)

model_fda = ResNet50Classifier(num_classes=7)
model_fda = train_classifier(model_fda, fda_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc = evaluate_classifier(model_fda, target_test_loader, device=device)
results['FDA'] = acc

### Results Summary

In [ ]:
print_results_table(results, 'Photo -> Sketch (PACS)')
save_results(results, 'Photo_to_Sketch', save_dir=SAVE_DIR)